# Hurdle Model: Alternative Approach to Selection Bias

**Problem:** Naive zero-imputation failed (corr_mse +31% worse).

**Alternative Approach: Hurdle Model (Two-Stage)**

Instead of treating zeros as regular observations, we model the data generation process:

1. **Stage 1 (Classification):** P(demand > 0 | X, price)
   - Binary: will there be any sales this week?
   - This captures the "extensive margin" of price response

2. **Stage 2 (Regression):** E[demand | demand > 0, X, price]
   - Conditional on having sales, how many?
   - This captures the "intensive margin"

**Combined prediction:**
$$E[demand] = P(demand > 0) \times E[demand | demand > 0]$$

**Elasticity decomposition:**
$$\varepsilon_{total} = \varepsilon_{extensive} + \varepsilon_{intensive}$$

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')
print("Setup complete")

Setup complete


## 1. Load Data with Zeros

In [2]:
# Load panel with zeros
panel = pd.read_csv(DATA_DIR / 'panel_with_zeros.csv')
panel['week'] = pd.to_datetime(panel['week'])

# Encode category
le_cat = LabelEncoder()
panel['category_encoded'] = le_cat.fit_transform(
    panel['product_category_name_english'].fillna('unknown')
)

# Create binary target for Stage 1
panel['has_sales'] = (panel['demand'] > 0).astype(int)

print(f"Total rows: {len(panel):,}")
print(f"Zero-demand rows: {(panel['demand']==0).sum():,} ({(panel['demand']==0).mean():.1%})")
print(f"Positive-demand rows: {(panel['demand']>0).sum():,} ({(panel['demand']>0).mean():.1%})")

Total rows: 54,742
Zero-demand rows: 36,772 (67.2%)
Positive-demand rows: 17,970 (32.8%)


In [3]:
# Features (same as before)
feature_cols = [
    'r', 'r_clipped', 'avg_price', 'P0',
    'price_lag_1', 'price_roll_4', 'price_std', 'price_range', 'min_price', 'max_price',
    'year', 'month', 'weekofyear', 'week_sin', 'week_cos',
    'demand_lag_1', 'demand_lag_2', 'demand_roll_4', 'weeks_since_last_sale',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
    'product_photos_qty', 'product_name_length', 'product_description_length',
    'sku_review_count', 'sku_review_mean', 'sku_share_low',
    'category_encoded'
]

# Fill NaN
for col in feature_cols:
    if col in panel.columns:
        panel[col] = panel[col].fillna(0)

# Split masks
train_mask = panel['split'] == 'train'
val_mask = panel['split'] == 'val'
test_mask = panel['split'] == 'test'

print(f"Train: {train_mask.sum():,}, Val: {val_mask.sum():,}, Test: {test_mask.sum():,}")

Train: 37,175, Val: 9,677, Test: 7,890


## 2. Stage 1: Classification (P(demand > 0))

In [4]:
# Prepare data for Stage 1
X_train_s1 = panel.loc[train_mask, feature_cols]
y_train_s1 = panel.loc[train_mask, 'has_sales']
X_val_s1 = panel.loc[val_mask, feature_cols]
y_val_s1 = panel.loc[val_mask, 'has_sales']
X_test_s1 = panel.loc[test_mask, feature_cols]
y_test_s1 = panel.loc[test_mask, 'has_sales']

print(f"Stage 1 class balance (train):")
print(f"  No sales: {(y_train_s1==0).sum():,} ({(y_train_s1==0).mean():.1%})")
print(f"  Has sales: {(y_train_s1==1).sum():,} ({(y_train_s1==1).mean():.1%})")

Stage 1 class balance (train):
  No sales: 25,467 (68.5%)
  Has sales: 11,708 (31.5%)


In [5]:
# Train Stage 1 classifier
dtrain_s1 = xgb.DMatrix(X_train_s1, label=y_train_s1)
dval_s1 = xgb.DMatrix(X_val_s1, label=y_val_s1)
dtest_s1 = xgb.DMatrix(X_test_s1, label=y_test_s1)

# Handle class imbalance
scale_pos_weight = (y_train_s1 == 0).sum() / (y_train_s1 == 1).sum()

xgb_params_s1 = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'seed': 42
}

print(f"Training Stage 1 (Classification)...")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

bst_s1 = xgb.train(
    xgb_params_s1,
    dtrain_s1,
    num_boost_round=300,
    evals=[(dtrain_s1, 'train'), (dval_s1, 'val')],
    early_stopping_rounds=30,
    verbose_eval=50
)

print(f"\nBest iteration: {bst_s1.best_iteration}")

Training Stage 1 (Classification)...
scale_pos_weight: 2.18


[0]	train-auc:0.76034	val-auc:0.71832


[50]	train-auc:0.79052	val-auc:0.73257


[100]	train-auc:0.81747	val-auc:0.73901


[150]	train-auc:0.84040	val-auc:0.74476


[200]	train-auc:0.85600	val-auc:0.74445


[216]	train-auc:0.85988	val-auc:0.74592



Best iteration: 186


In [6]:
# Stage 1 predictions
p_sales_train = bst_s1.predict(dtrain_s1)
p_sales_val = bst_s1.predict(dval_s1)
p_sales_test = bst_s1.predict(dtest_s1)

# Metrics
train_auc = roc_auc_score(y_train_s1, p_sales_train)
val_auc = roc_auc_score(y_val_s1, p_sales_val)
test_auc = roc_auc_score(y_test_s1, p_sales_test)

print("="*60)
print("STAGE 1: Classification Performance")
print("="*60)
print(f"\n{'Set':<10} {'AUC':>10}")
print("-" * 22)
print(f"{'Train':<10} {train_auc:>10.4f}")
print(f"{'Val':<10} {val_auc:>10.4f}")
print(f"{'Test':<10} {test_auc:>10.4f}")

STAGE 1: Classification Performance

Set               AUC
----------------------
Train          0.8599
Val            0.7459
Test           0.7267


## 3. Stage 2: Regression (demand | demand > 0)

In [7]:
# Filter to positive demand only
train_pos = panel[train_mask & (panel['demand'] > 0)]
val_pos = panel[val_mask & (panel['demand'] > 0)]
test_pos = panel[test_mask & (panel['demand'] > 0)]

X_train_s2 = train_pos[feature_cols]
y_train_s2 = train_pos['y']  # log(demand + 1)
X_val_s2 = val_pos[feature_cols]
y_val_s2 = val_pos['y']
X_test_s2 = test_pos[feature_cols]
y_test_s2 = test_pos['y']

print(f"Stage 2 data (positive demand only):")
print(f"  Train: {len(X_train_s2):,}")
print(f"  Val: {len(X_val_s2):,}")
print(f"  Test: {len(X_test_s2):,}")

Stage 2 data (positive demand only):
  Train: 11,708
  Val: 3,392
  Test: 2,870


In [8]:
# Train Stage 2 regressor
dtrain_s2 = xgb.DMatrix(X_train_s2, label=y_train_s2)
dval_s2 = xgb.DMatrix(X_val_s2, label=y_val_s2)
dtest_s2 = xgb.DMatrix(X_test_s2, label=y_test_s2)

xgb_params_s2 = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'seed': 42
}

print("Training Stage 2 (Regression on positive demand)...")

bst_s2 = xgb.train(
    xgb_params_s2,
    dtrain_s2,
    num_boost_round=500,
    evals=[(dtrain_s2, 'train'), (dval_s2, 'val')],
    early_stopping_rounds=50,
    verbose_eval=50
)

print(f"\nBest iteration: {bst_s2.best_iteration}")

Training Stage 2 (Regression on positive demand)...
[0]	train-rmse:0.41816	val-rmse:0.44975


[50]	train-rmse:0.29748	val-rmse:0.35937


[75]	train-rmse:0.28507	val-rmse:0.36223



Best iteration: 25


## 4. Combined Hurdle Model Predictions

In [9]:
def compute_corr_mse(y_true, y_pred, groups):
    """Compute group-corrected MSE."""
    df = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'group': groups})
    
    def group_mse(g):
        if len(g) < 2:
            return 0, 0
        y_t = g['y_true'].values
        y_p = g['y_pred'].values
        y_t_c = y_t - y_t.mean()
        y_p_c = y_p - y_p.mean()
        mse = ((y_t_c - y_p_c) ** 2).mean()
        return mse * len(g), len(g)
    
    results = df.groupby('group').apply(group_mse)
    total = sum(r[0] for r in results)
    weight = sum(r[1] for r in results)
    return total / weight if weight > 0 else 0

In [10]:
# Combined prediction: E[demand] = P(sales) * E[demand | sales]
# On log scale: log(E[demand]) ≈ log(P(sales)) + log(E[demand | sales])

# For test set:
# Get P(sales) for all test rows
p_sales = bst_s1.predict(dtest_s1)

# Get E[y | sales] for all test rows (even zeros, for prediction)
X_test_all = panel.loc[test_mask, feature_cols]
dtest_all = xgb.DMatrix(X_test_all)
y_cond = bst_s2.predict(dtest_all)

# Combined prediction on log scale
# E[demand] = P(sales) * exp(y_cond) - but we predict on log scale
# So: y_hurdle = log(P(sales) * (exp(y_cond) - 1) + 1) ≈ log(P(sales)) + y_cond for small P

# More accurate: E[y] = P(s) * y_cond + (1-P(s)) * 0
y_hurdle = p_sales * y_cond

# Alternative: threshold-based
threshold = 0.5
y_hurdle_thresh = np.where(p_sales > threshold, y_cond, 0)

y_test_all = panel.loc[test_mask, 'y'].values
product_ids_test = panel.loc[test_mask, 'product_id'].values

In [11]:
# Compute metrics
corr_mse_hurdle = compute_corr_mse(y_test_all, y_hurdle, product_ids_test)
corr_mse_hurdle_thresh = compute_corr_mse(y_test_all, y_hurdle_thresh, product_ids_test)

# Also compute for observed-only (fair comparison)
observed_mask = panel.loc[test_mask, 'is_imputed'] == 0
y_test_obs = y_test_all[observed_mask.values]
y_hurdle_obs = y_hurdle[observed_mask.values]
product_ids_obs = product_ids_test[observed_mask.values]

corr_mse_hurdle_obs = compute_corr_mse(y_test_obs, y_hurdle_obs, product_ids_obs)

print("="*60)
print("HURDLE MODEL RESULTS")
print("="*60)

print(f"\n{'Model':<45} {'corr_mse':>10}")
print("-" * 57)
print(f"{'Baseline XGBoost (original, no zeros)':<45} {'0.0964':>10}")
print(f"{'Naive zeros XGBoost (all rows)':<45} {'0.1814':>10}")
print(f"{'Naive zeros XGBoost (observed only)':<45} {'0.1263':>10}")
print("-" * 57)
print(f"{'Hurdle Model (P*y, all rows)':<45} {corr_mse_hurdle:>10.4f}")
print(f"{'Hurdle Model (threshold, all rows)':<45} {corr_mse_hurdle_thresh:>10.4f}")
print(f"{'Hurdle Model (P*y, observed only)':<45} {corr_mse_hurdle_obs:>10.4f}")

HURDLE MODEL RESULTS

Model                                           corr_mse
---------------------------------------------------------
Baseline XGBoost (original, no zeros)             0.0964
Naive zeros XGBoost (all rows)                    0.1814
Naive zeros XGBoost (observed only)               0.1263
---------------------------------------------------------
Hurdle Model (P*y, all rows)                      0.1798
Hurdle Model (threshold, all rows)                0.2460
Hurdle Model (P*y, observed only)                 0.1194


## 5. Elasticity Analysis: Extensive vs Intensive Margin

In [12]:
def estimate_elasticity(model, X_base, r_col='r', delta=0.01, is_classifier=False):
    """Estimate elasticity using finite differences."""
    X_low = X_base.copy()
    X_high = X_base.copy()
    
    X_low[r_col] = X_base[r_col] - delta
    X_high[r_col] = X_base[r_col] + delta
    
    d_low = xgb.DMatrix(X_low)
    d_high = xgb.DMatrix(X_high)
    
    y_low = model.predict(d_low)
    y_high = model.predict(d_high)
    
    if is_classifier:
        # For P(sales), elasticity = d log(P) / d log(r) = (dP/P) / (dr/r)
        # ≈ (P_high - P_low) / P_base / (2*delta / r_base)
        d_base = xgb.DMatrix(X_base)
        y_base = model.predict(d_base)
        y_base = np.clip(y_base, 0.01, 0.99)  # avoid division by zero
        r_base = X_base[r_col].values
        elasticity = ((y_high - y_low) / y_base) / (2 * delta / r_base)
    else:
        # For regression, dy/dr
        elasticity = (y_high - y_low) / (2 * delta)
    
    return elasticity

In [13]:
# Compute elasticities for test set (observed rows only)
X_test_obs_df = X_test_all[observed_mask.values].reset_index(drop=True)

# Extensive margin: how price affects P(sales)
elasticity_extensive = estimate_elasticity(bst_s1, X_test_obs_df, is_classifier=True)

# Intensive margin: how price affects demand | sales
elasticity_intensive = estimate_elasticity(bst_s2, X_test_obs_df, is_classifier=False)

print("="*60)
print("ELASTICITY DECOMPOSITION")
print("="*60)

print(f"\n### Extensive Margin (effect on P(sales))")
print(f"  Mean: {np.mean(elasticity_extensive):.4f}")
print(f"  Median: {np.median(elasticity_extensive):.4f}")
print(f"  Interpretation: {abs(np.mean(elasticity_extensive)):.1%} change in P(sales) per 1% price increase")

print(f"\n### Intensive Margin (effect on demand | sales)")
print(f"  Mean: {np.mean(elasticity_intensive):.4f}")
print(f"  Median: {np.median(elasticity_intensive):.4f}")
print(f"  Interpretation: {abs(np.mean(elasticity_intensive)):.2f} log-points demand change per unit r change")

print(f"\n### Comparison with Baseline")
print(f"  Original XGBoost elasticity: ~-0.5")
print(f"  Naive zeros elasticity: -0.20")
print(f"  Hurdle intensive margin: {np.mean(elasticity_intensive):.4f}")

ELASTICITY DECOMPOSITION

### Extensive Margin (effect on P(sales))
  Mean: 0.0149
  Median: 0.0000
  Interpretation: 1.5% change in P(sales) per 1% price increase

### Intensive Margin (effect on demand | sales)
  Mean: -0.2353
  Median: 0.0000
  Interpretation: 0.24 log-points demand change per unit r change

### Comparison with Baseline
  Original XGBoost elasticity: ~-0.5
  Naive zeros elasticity: -0.20
  Hurdle intensive margin: -0.2353


In [14]:
# Distribution of elasticities
print("\n### Intensive Margin Elasticity Distribution")
percentiles = [5, 25, 50, 75, 95]
for p in percentiles:
    val = np.percentile(elasticity_intensive, p)
    print(f"  {p}th percentile: {val:.4f}")

# How many have "reasonable" elasticity?
reasonable = (elasticity_intensive < -0.5) & (elasticity_intensive > -5)
print(f"\n  Products with elasticity in [-5, -0.5]: {reasonable.sum()} ({reasonable.mean():.1%})")


### Intensive Margin Elasticity Distribution
  5th percentile: -2.2555
  25th percentile: -0.3476
  50th percentile: 0.0000
  75th percentile: 0.0000
  95th percentile: 1.0786

  Products with elasticity in [-5, -0.5]: 585 (20.4%)


## 6. Summary and Comparison

In [15]:
print("\n" + "="*70)
print("FINAL SUMMARY: SELECTION BIAS EXPERIMENTS")
print("="*70)

print("""
### Approaches Tested:

1. **Baseline XGBoost** (original data, no zeros)
   - corr_mse: 0.0964
   - Elasticity: ~-0.5

2. **Naive Zero Imputation** (add zeros with forward-filled prices)
   - corr_mse (all): 0.1814 (+88% worse)
   - corr_mse (observed): 0.1263 (+31% worse)
   - Elasticity: -0.20 (closer to zero - WRONG direction)
   - FAILED: Zero inflation hurt the model

3. **Hurdle Model** (separate classification + regression)
   - Stage 1 AUC: see above
   - corr_mse: see above
   - Provides elasticity decomposition
""")

print(f"\n### Key Insight:")
print(f"The Hurdle model separates:")
print(f"  - EXTENSIVE margin: Does price affect WHETHER there's a sale?")
print(f"  - INTENSIVE margin: Does price affect HOW MUCH is sold given a sale?")
print(f"\nThis decomposition is more economically meaningful than a single elasticity.")


FINAL SUMMARY: SELECTION BIAS EXPERIMENTS

### Approaches Tested:

1. **Baseline XGBoost** (original data, no zeros)
   - corr_mse: 0.0964
   - Elasticity: ~-0.5

2. **Naive Zero Imputation** (add zeros with forward-filled prices)
   - corr_mse (all): 0.1814 (+88% worse)
   - corr_mse (observed): 0.1263 (+31% worse)
   - Elasticity: -0.20 (closer to zero - WRONG direction)
   - FAILED: Zero inflation hurt the model

3. **Hurdle Model** (separate classification + regression)
   - Stage 1 AUC: see above
   - corr_mse: see above
   - Provides elasticity decomposition


### Key Insight:
The Hurdle model separates:
  - EXTENSIVE margin: Does price affect WHETHER there's a sale?
  - INTENSIVE margin: Does price affect HOW MUCH is sold given a sale?

This decomposition is more economically meaningful than a single elasticity.


## 7. Documented Results: Selection Bias Experiments

### Summary Table

| Approach | corr_mse (observed) | Elasticity | Status |
|----------|---------------------|------------|--------|
| Baseline XGBoost | **0.0964** | -0.50 | Best |
| Naive Zero Imputation | 0.1263 (+31%) | -0.20 | Failed |
| Hurdle Model | 0.1194 (+24%) | -0.24 (intensive) | Partial |

### Key Findings

1. **Naive zero imputation failed** — adding imputed zeros with forward-filled prices degraded model performance

2. **Hurdle model slightly better** — corr_mse 0.1194 vs 0.1263 for naive zeros, but still worse than baseline

3. **Elasticity decomposition** provides economic insight:
   - Extensive margin (price → P(sales)): ~0 (price doesn't affect whether there's a sale)
   - Intensive margin (price → demand|sales): -0.24 (modest effect on quantity)

4. **20% of products** show "reasonable" elasticity in [-5, -0.5] range

### Implications for Thesis

**Selection bias correction didn't help because:**
1. Forward-filled prices are noisy — we don't know actual listing prices during zero-sale weeks
2. The dataset may not have severe selection bias — products are observed fairly consistently
3. Price variation is limited — hard to identify elasticity regardless of approach

**Recommendation:** 
- Use the **original baseline XGBoost (corr_mse = 0.0964)** as the primary benchmark
- Document selection bias experiments as robustness checks (negative results)
- Focus on the Two-Head NN comparison with baseline XGBoost